In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

In [5]:
#Load data
df = pd.read_csv('/Users/preciousajilore/Documents/GitHub/torchmtlr/notebooks/x_before_surgery.csv')

In [6]:
#Drop the rows where failure is equal to 2
df = df[df['failure'] != 2]

In [ ]:
#Sanityy check lol
df['failure'].unique()

array([1, 0])

In [8]:
df.columns

Index(['Unnamed: 0', 'distal', 'penile', 'stxlength', '#strictures',
       'charlsons', 'cormorbidity', 'diabetes', 'copd', 'smoker', 'bmi35+',
       'bmiexact', 'prevprocedure', '#prevprocedures', 'cysto', 'open',
       'ordate', 'urine', 'failure', 'patent', 'satisfaction',
       'datetofailureorfollowup', 'Date of Surgery', 'fu', 'time_to_event',
       'open_clean', 'stxlength_1', 'stxlength_2', 'stxlocation_0',
       'stxlocation_1', 'stxlocation_2', 'stxlocation_3', 'stxlocation_4',
       'stxlocation_5', 'stxlocation_6', 'stxetiology_0', 'stxetiology_1',
       'stxetiology_18', 'stxetiology_2', 'stxetiology_3', 'stxetiology_4',
       'stxetiology_5', 'stxetiology_6'],
      dtype='object')

In [9]:
#Drop columns that we wont use for prediction

"""

Index(['Unnamed: 0', 'distal', 'penile', 'stxlength', '#strictures',
       'charlsons', 'cormorbidity', 'diabetes', 'copd', 'smoker', 'bmi35+',
       'bmiexact', 'prevprocedure', '#prevprocedures', 'cysto', 'open',
       'ordate', 'urine', 'failure', 'patent', 'satisfaction',
       'datetofailureorfollowup', 'Date of Surgery', 'fu', 'time_to_event',
       'open_clean', 'stxlength_1', 'stxlength_2', 'stxlocation_0',
       'stxlocation_1', 'stxlocation_2', 'stxlocation_3', 'stxlocation_4',
       'stxlocation_5', 'stxlocation_6', 'stxetiology_0', 'stxetiology_1',
       'stxetiology_18', 'stxetiology_2', 'stxetiology_3', 'stxetiology_4',
       'stxetiology_5', 'stxetiology_6'],
      dtype='object')

"""
drop = ['Unnamed: 0','ordate','datetofailureorfollowup', 'Date of Surgery', 'bmiexact']

df = df.drop(drop, axis=1)

In [ ]:
#Sanity check 
df.columns

Index(['distal', 'penile', 'stxlength', '#strictures', 'charlsons',
       'cormorbidity', 'diabetes', 'copd', 'smoker', 'bmi35+', 'prevprocedure',
       '#prevprocedures', 'cysto', 'open', 'urine', 'failure', 'patent',
       'satisfaction', 'fu', 'time_to_event', 'open_clean', 'stxlength_1',
       'stxlength_2', 'stxlocation_0', 'stxlocation_1', 'stxlocation_2',
       'stxlocation_3', 'stxlocation_4', 'stxlocation_5', 'stxlocation_6',
       'stxetiology_0', 'stxetiology_1', 'stxetiology_18', 'stxetiology_2',
       'stxetiology_3', 'stxetiology_4', 'stxetiology_5', 'stxetiology_6'],
      dtype='object')

In [11]:
# Seperate the features and label
X = df.drop('failure', axis=1)
Y =  df['failure']


In [12]:
#Just incase we have categorical variables, we need to convert them to numerical variables using one hot encoding
X = pd.get_dummies(X, drop_first=True)

In [13]:
#Split into train and test sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)  

In [ ]:
#Standarize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)y

In [18]:
#Define the models we want to use
models = {
   'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
   'RandomForest': RandomForestClassifier(random_state=42),
   'SVM': SVC(kernel = 'rbf', probability= True, random_state=42),
   'MLP Neural Net': MLPClassifier(hidden_layer_sizes = (32,16), max_iter= 300, random_state=42),
}

In [19]:
#Train and evaluate each model

for name, model in models.items():
    if name in ['SVM', 'MLP Neural Net']:
        model.fit(X_train_scaled, Y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, Y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(Y_test, y_pred)
    precision = precision_score(Y_test, y_pred)
    recall = recall_score(Y_test, y_pred)
    f1 = f1_score(Y_test, y_pred)
    auc = roc_auc_score(Y_test, y_prob)

    #Print metrics
    print(f"{name} Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC Score: {auc:.4f}")
    print(classification_report(Y_test, y_pred, digits=3))


LogisticRegression Metrics:
Accuracy: 0.9706
Precision: 0.8824
Recall: 0.8824
F1 Score: 0.8824
AUC Score: 0.9886
              precision    recall  f1-score   support

           0      0.983     0.983     0.983       119
           1      0.882     0.882     0.882        17

    accuracy                          0.971       136
   macro avg      0.933     0.933     0.933       136
weighted avg      0.971     0.971     0.971       136

RandomForest Metrics:
Accuracy: 0.9559
Precision: 0.9231
Recall: 0.7059
F1 Score: 0.8000
AUC Score: 0.9941
              precision    recall  f1-score   support

           0      0.959     0.992     0.975       119
           1      0.923     0.706     0.800        17

    accuracy                          0.956       136
   macro avg      0.941     0.849     0.888       136
weighted avg      0.955     0.956     0.953       136

SVM Metrics:
Accuracy: 0.9191
Precision: 1.0000
Recall: 0.3529
F1 Score: 0.5217
AUC Score: 0.9891
              precision    r